In [1]:
import json

candidates = []

with open("F:\\autorecruit-\\data_forensic _files\\candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

In [3]:
import pandas as pd
skills_rows = []

for c in candidates:

    cid = c["candidate_id"]

    for skill in c["skills"]:

        skills_rows.append({
            "candidate_id": cid,
            "skill": skill["name"],
            "proficiency": skill["proficiency"],
            "duration_months": skill["duration_months"],
            "endorsements": skill["endorsements"]
        })

skills_df = pd.DataFrame(skills_rows)

print(skills_df.shape)
skills_df.head()

(960302, 5)


,candidate_id,skill,proficiency,duration_months,endorsements
0,CAND_0000001,Tailwind,intermediate,13,3
1,CAND_0000001,NLP,advanced,26,37
2,CAND_0000001,Image Classification,advanced,40,7
3,CAND_0000001,Fine-tuning LLMs,advanced,36,21
4,CAND_0000001,Weights & Biases,intermediate,30,13


In [4]:
candidate_corpus = {}

for c in candidates:

    cid = c["candidate_id"]

    summary = c["profile"]["summary"]

    headline = c["profile"]["headline"]

    descriptions = " ".join([
        job["description"]
        for job in c["career_history"]
    ])

    candidate_corpus[cid] = " ".join([
        headline,
        summary,
        descriptions
    ])

In [6]:
candidate_corpus

{'CAND_0000001': "Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice. Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure f

In [8]:
evidence_rows = []

for c in candidates:

    cid = c["candidate_id"]

    corpus = candidate_corpus[cid].lower()

    total_skills = 0
    supported_skills = 0

    for skill in c["skills"]:

        total_skills += 1

        if skill["name"].lower() in corpus:
            supported_skills += 1

    evidence_rows.append({
        "candidate_id": cid,
        "skills_claimed": total_skills,
        "skills_supported": supported_skills
    })

evidence_df = pd.DataFrame(evidence_rows)
print(evidence_df.shape)
evidence_df.head()

(100000, 3)


,candidate_id,skills_claimed,skills_supported
0,CAND_0000001,17,0
1,CAND_0000002,9,1
2,CAND_0000003,6,1
3,CAND_0000004,10,1
4,CAND_0000005,6,0


In [10]:
evidence_df["evidence_ratio"] = (
    evidence_df["skills_supported"]
    /
    evidence_df["skills_claimed"]
)

evidence_df[["candidate_id", "skills_claimed", "skills_supported", "evidence_ratio"]].head()

,candidate_id,skills_claimed,skills_supported,evidence_ratio
0,CAND_0000001,17,0,0.000000
1,CAND_0000002,9,1,0.111111
2,CAND_0000003,6,1,0.166667
3,CAND_0000004,10,1,0.100000
4,CAND_0000005,6,0,0.000000


In [12]:
evidence_df.sort_values(
    "evidence_ratio"
)

,candidate_id,skills_claimed,skills_supported,evidence_ratio
46,CAND_0000047,7,0,0.000000
54512,CAND_0054513,7,0,0.000000
54511,CAND_0054512,5,0,0.000000
54509,CAND_0054510,8,0,0.000000
54507,CAND_0054508,11,0,0.000000
...,...,...,...,...
46378,CAND_0046379,8,5,0.625000
5126,CAND_0005127,6,4,0.666667
49829,CAND_0049830,6,4,0.666667
44891,CAND_0044892,6,4,0.666667


In [13]:
inflation_rows = []

for c in candidates:

    cid = c["candidate_id"]

    corpus = candidate_corpus[cid].lower()

    inflated = 0

    for skill in c["skills"]:

        prof = skill["proficiency"]

        found = (
            skill["name"].lower()
            in corpus
        )

        if prof == "advanced" and not found:
            inflated += 1

    inflation_rows.append({
        "candidate_id": cid,
        "inflated_skills": inflated
    })

inflation_df = pd.DataFrame(
    inflation_rows
)

In [14]:
inflation_df

,candidate_id,inflated_skills
0,CAND_0000001,7
1,CAND_0000002,0
2,CAND_0000003,0
3,CAND_0000004,0
4,CAND_0000005,1
...,...,...
99995,CAND_0099996,0
99996,CAND_0099997,0
99997,CAND_0099998,1
99998,CAND_0099999,1


In [15]:
jd_keywords = [

    "retrieval",

    "ranking",

    "embedding",

    "vector",

    "python",

    "evaluation",

    "ndcg",

    "mrr",

    "map",

    "search",

    "recommendation"
]

In [17]:
jd_rows = []

for cid, corpus in candidate_corpus.items():

    text = corpus.lower()

    hits = 0

    for kw in jd_keywords:

        if kw in text:
            hits += 1

    jd_rows.append({
        "candidate_id": cid,
        "jd_evidence_hits": hits
    })

jd_df = pd.DataFrame(jd_rows)

jd_df

,candidate_id,jd_evidence_hits
0,CAND_0000001,1
1,CAND_0000002,1
2,CAND_0000003,0
3,CAND_0000004,1
4,CAND_0000005,0
...,...,...
99995,CAND_0099996,1
99996,CAND_0099997,1
99997,CAND_0099998,1
99998,CAND_0099999,1


In [18]:
merged = (
    evidence_df
    .merge(
        inflation_df,
        on="candidate_id"
    )
)

In [20]:
merged

,candidate_id,skills_claimed,skills_supported,evidence_ratio,inflated_skills,skill_credibility_score
0,CAND_0000001,17,0,0.000000,7,-0.411765
1,CAND_0000002,9,1,0.111111,0,0.111111
2,CAND_0000003,6,1,0.166667,0,0.166667
3,CAND_0000004,10,1,0.100000,0,0.100000
4,CAND_0000005,6,0,0.000000,1,-0.166667
...,...,...,...,...,...,...
99995,CAND_0099996,10,1,0.100000,0,0.100000
99996,CAND_0099997,7,1,0.142857,0,0.142857
99997,CAND_0099998,10,0,0.000000,1,-0.100000
99998,CAND_0099999,7,0,0.000000,1,-0.142857


In [19]:
merged[
    "skill_credibility_score"
] = (

    merged["evidence_ratio"]

    -

    (
        merged["inflated_skills"]
        /
        merged["skills_claimed"]
    )
)

In [23]:
merged_df = merged[merged["skill_credibility_score"] > 0]

merged_df

,candidate_id,skills_claimed,skills_supported,evidence_ratio,inflated_skills,skill_credibility_score
1,CAND_0000002,9,1,0.111111,0,0.111111
2,CAND_0000003,6,1,0.166667,0,0.166667
3,CAND_0000004,10,1,0.100000,0,0.100000
8,CAND_0000009,6,1,0.166667,0,0.166667
16,CAND_0000017,8,1,0.125000,0,0.125000
...,...,...,...,...,...,...
99975,CAND_0099976,7,1,0.142857,0,0.142857
99984,CAND_0099985,7,1,0.142857,0,0.142857
99993,CAND_0099994,16,1,0.062500,0,0.062500
99995,CAND_0099996,10,1,0.100000,0,0.100000


In [25]:
merged_df.sort_values(
    "skill_credibility_score",
    ascending=False
)

,candidate_id,skills_claimed,skills_supported,evidence_ratio,inflated_skills,skill_credibility_score
5126,CAND_0005127,6,4,0.666667,0,0.666667
49829,CAND_0049830,6,4,0.666667,0,0.666667
15744,CAND_0015745,6,4,0.666667,0,0.666667
44891,CAND_0044892,6,4,0.666667,0,0.666667
4725,CAND_0004726,10,6,0.600000,0,0.600000
...,...,...,...,...,...,...
46433,CAND_0046434,21,3,0.142857,2,0.047619
79037,CAND_0079038,22,4,0.181818,3,0.045455
49886,CAND_0049887,22,3,0.136364,2,0.045455
71244,CAND_0071245,22,3,0.136364,2,0.045455


In [26]:
intent_rows = []

for c in candidates:

    s = c["redrob_signals"]

    intent_rows.append({

        "candidate_id":
        c["candidate_id"],

        "open_to_work":
        s["open_to_work_flag"],

        "applications":
        s["applications_submitted_30d"],

        "response_rate":
        s["recruiter_response_rate"],

        "notice_period":
        s["notice_period_days"]
    })

In [30]:
intent_rows

[{'candidate_id': 'CAND_0000001',
  'open_to_work': True,
  'applications': 2,
  'response_rate': 0.34,
  'notice_period': 60},
 {'candidate_id': 'CAND_0000002',
  'open_to_work': True,
  'applications': 1,
  'response_rate': 0.29,
  'notice_period': 60},
 {'candidate_id': 'CAND_0000003',
  'open_to_work': False,
  'applications': 9,
  'response_rate': 0.46,
  'notice_period': 150},
 {'candidate_id': 'CAND_0000004',
  'open_to_work': False,
  'applications': 9,
  'response_rate': 0.26,
  'notice_period': 120},
 {'candidate_id': 'CAND_0000005',
  'open_to_work': True,
  'applications': 2,
  'response_rate': 0.37,
  'notice_period': 30},
 {'candidate_id': 'CAND_0000006',
  'open_to_work': False,
  'applications': 8,
  'response_rate': 0.12,
  'notice_period': 150},
 {'candidate_id': 'CAND_0000007',
  'open_to_work': False,
  'applications': 1,
  'response_rate': 0.62,
  'notice_period': 30},
 {'candidate_id': 'CAND_0000008',
  'open_to_work': False,
  'applications': 5,
  'response_rate'

In [36]:
def intent_score(row):

    score = 0

    if row["open_to_work"]:
        score += 40

    score += (
        min(
            row["applications"],
            10
        ) * 3
    )

    score += (
        row["response_rate"]
        * 30
    )

    return score

intent_scores = []

print(intent_rows[4])
# intent_df = pd.DataFrame(intent_rows)
# intent_df

{'candidate_id': 'CAND_0000005', 'open_to_work': True, 'applications': 2, 'response_rate': 0.37, 'notice_period': 30}
